In [ ]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from pycaret.classification import setup, compare_models, predict_model, pull
import h2o
from h2o.automl import H2OAutoML
from flaml import AutoML

In [14]:
%pip install "pandas==2.1.4" "numpy==1.26.4" "scipy==1.11.4" "joblib==1.3.2" "scikit-learn==1.4.2" "pycaret==3.3.2"


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
df = pd.read_csv("../data/creditcard.csv")

In [ ]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR PYCARET
# =========================
# %pip install pycaret psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PYCARET + MÉTRICA COMPUTACIONAL (TREINO)
    # =========================
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print("Iniciando treinamento do PyCaret (setup + compare_models)...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    exp = setup(
        data=train_df,
        test_data=test_df,
        target="Class",
        session_id=SEED,
        verbose=False,
        fix_imbalance=False,
    )

    # Retorna os 10 melhores modelos treinados
    modelos_treinados = compare_models(n_select=10)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # LEADERBOARD INTERNO DO PYCARET
    # =========================
    print("\n===== LEADERBOARD (Validação PyCaret) =====")
    print(pull())
    leaderboard = pull()

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS (No Test Set)
    # =========================
    resultados = []

    for model in modelos_treinados:
        model_name = model.__class__.__name__

        # Tempo de inferência no conjunto de teste (previsão + score)
        inicio_inferencia = time.perf_counter()
        preds_df = predict_model(model, data=test_df, verbose=False)
        tempo_inferencia = time.perf_counter() - inicio_inferencia

        y_true = test_df['Class'].values
        y_pred = preds_df['prediction_label'].astype(int).values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

        score_col = [col for col in preds_df.columns if 'score' in col]
        if score_col:
            pos_score_col = [c for c in score_col if '1' in str(c)]
            if pos_score_col:
                y_score = preds_df[pos_score_col[0]].values
            else:
                y_score = preds_df[score_col[0]].values
        else:
            y_score = y_pred

        average_precision = round(average_precision_score(y_true, y_score), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name,
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score Weighted": f1_weighted,
            "F1-score": f1,
            "average_precision": average_precision,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADOS
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS (SEED {}) =====".format(SEED))
    print(df_resultados.sort_values(by="average_precision", ascending=False).to_string(index=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9995,0.9734,0.8246,0.8660,0.8432,0.8429,0.8440,173.2780
et,Extra Trees Classifier,0.9995,0.9780,0.8271,0.8858,0.8537,0.8535,0.8549,132.3710
lightgbm,Light Gradient Boosting Machine,0.9990,0.9653,0.8221,0.6740,0.7380,0.7375,0.7425,127.4070
dt,Decision Tree Classifier,0.9978,0.8911,0.7840,0.4250,0.5501,0.5491,0.5757,199.8970
gbc,Gradient Boosting Classifier,0.9939,0.9732,0.8651,0.2039,0.3297,0.3278,0.4180,210.9140
nb,Naive Bayes,0.9924,0.9654,0.7485,0.1540,0.2553,0.2531,0.3372,252.7340
lr,Logistic Regression,0.9893,0.9800,0.8854,0.1284,0.2240,0.2216,0.3345,101.9190
ada,Ada Boost Classifier,0.9884,0.9778,0.8754,0.1183,0.2082,0.2058,0.3190,143.3010
ridge,Ridge Classifier,0.9869,0.9727,0.8223,0.1001,0.1784,0.1758,0.2841,126.0070
lda,Linear Discriminant Analysis,0.9869,0.9727,0.8223,0.1001,0.1784,0.1758,0.2841,125.6360



===== LEADERBOARD (Validação PyCaret) =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9995  0.9734  0.8246  0.8660   
et                 Extra Trees Classifier    0.9995  0.9780  0.8271  0.8858   
lightgbm  Light Gradient Boosting Machine    0.9990  0.9653  0.8221  0.6740   
dt               Decision Tree Classifier    0.9978  0.8911  0.7840  0.4250   
gbc          Gradient Boosting Classifier    0.9939  0.9732  0.8651  0.2039   
nb                            Naive Bayes    0.9924  0.9654  0.7485  0.1540   
lr                    Logistic Regression    0.9893  0.9800  0.8854  0.1284   
ada                  Ada Boost Classifier    0.9884  0.9778  0.8754  0.1183   
ridge                    Ridge Classifier    0.9869  0.9727  0.8223  0.1001   
lda          Linear Discriminant Analysis    0.9869  0.9727  0.8223  0.1001   
qda       Quadratic Discriminant Analysis    0.9748  0.9687  0.8779  0.0575   
knn    

In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR H2O
# =========================
# %pip install h2o psutil

# =========================
# INICIAR H2O
# =========================
h2o.init()

target_col = 'Class'

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PARA H2O
    # =========================
    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    train_h2o['Class'] = train_h2o['Class'].asfactor()
    test_h2o['Class'] = test_h2o['Class'].asfactor()

    x = train_h2o.columns[:-1]

    # =========================
    # AUTO ML + MÉTRICA COMPUTACIONAL (TREINO)
    # =========================
    aml = H2OAutoML(
        seed=SEED,
        balance_classes=False,
    )

    print("Iniciando treinamento do H2O AutoML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    aml.train(x=x, y=target_col, training_frame=train_h2o)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    print(aml.leaderboard)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    leaderboard_df = aml.leaderboard.as_data_frame()
    training_times = leaderboard_df.set_index("model_id")["training_time_ms"]

    resultados = []

    for model_id in leaderboard_df['model_id']:
        model = h2o.get_model(model_id)

        # Tempo de inferência para cada modelo
        inicio_inferencia = time.perf_counter()
        preds = model.predict(test_h2o).as_data_frame()
        tempo_inferencia = time.perf_counter() - inicio_inferencia
        
        y_true = test_df[target_col].values
        y_pred = preds['predict'].astype(int).values
        y_score = preds['p1'].values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_score), 4)
        
        tempo_treino_modelo_ms = training_times.get(model_id, 0)
        tempo_treino_modelo_s = round(tempo_treino_modelo_ms / 1000, 4) if pd.notna(tempo_treino_modelo_ms) else 0.0

        resultados.append({
            "Modelo": model_id,
            "Tipo": model.algo,
            "Seed": SEED,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "average_precision": average_precision,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Modelo (s)": tempo_treino_modelo_s,
            "Tempo Treino Total AutoML (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS SEED {} =====".format(SEED))
    print(df_resultados.sort_values(by="average_precision", ascending=False).to_string(index=False))

    h2o.remove_all()
    h2o.init()


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,24 days 3 hours 26 mins
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 18 days
H2O_cluster_name:,H2O_from_python_lannaces_gfr9u3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,698 Mb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
01:35:54.364: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

===== LEADERBOARD =====
model

: 

In [ ]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR FLAML
# =========================
# %pip install "flaml[automl]" psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # =========================
    # TREINAMENTO COM FLAML + MÉTRICA COMPUTACIONAL
    # =========================
    automl = AutoML()
    settings = {
        "task": "classification",
        "log_file_name": "flaml.log",
        "seed": SEED
    }

    print("Iniciando treinamento com FLAML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    automl.fit(X_train=X_train, y_train=y_train, **settings)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # RESULTADOS DO MELHOR MODELO
    # =========================
    print("\n===== MELHOR RESULTADO FLAML =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # SELECIONAR OS 5 MELHORES ESTIMADORES
    # =========================
    perdas_por_estimador = automl.best_loss_per_estimator
    melhores_estimadores = sorted(
        perdas_por_estimador.items(),
        key=lambda item: item[1]
    )[:5]

    if not melhores_estimadores:
        raise RuntimeError("O FLAML não encontrou estimadores avaliados.")

    print("\n===== 5 MELHORES ESTIMADORES (VALIDAÇÃO FLAML) =====")
    print(pd.DataFrame(
        melhores_estimadores,
        columns=["Modelo", "Loss validação"]
    ).to_string(index=False))

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (nome_modelo, loss_validacao) in enumerate(melhores_estimadores, start=1):
        modelo = automl.best_model_for_estimator(nome_modelo)
        
        # Tempo de inferência para cada modelo
        inicio_inferencia = time.perf_counter()
        y_pred = modelo.predict(X_test)

        if hasattr(modelo, "predict_proba"):
            probabilidades = modelo.predict_proba(X_test)
            classes = list(modelo.classes_)
            y_score = probabilidades[:, classes.index(1)]
        elif hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_test)
            if list(modelo.classes_).index(1) == 0:
                y_score = -y_score
        else:
            y_score = y_pred
            
        tempo_inferencia = time.perf_counter() - inicio_inferencia

        resultados.append({
            "Posição": rank,
            "Modelo": nome_modelo,
            "Loss validação": round(float(loss_validacao), 4),
            "Accuracy": round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Average Precision": round(average_precision_score(y_true, y_score), 4),
            "Seed": SEED,
            "Tempo Inferência (s)": round(tempo_inferencia, 4),
            "Tempo Treino Total (s)": tempo_treino_total,
            "Uso RAM Treino (MB)": mem_depois_treino
        })

    df_resultados_flaml = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====")
    print(df_resultados_flaml.to_string(index=False))


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Dataset carregado:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ... 